# 03 — Data Preprocessing & Feature Engineering

## Objective
Prepare the Bank Marketing dataset for machine learning by investigating
duplicate records, handling unknown values, reviewing feature types,
performing feature engineering, and defining a leakage-safe preprocessing
strategy.

## Key modelling considerations
- Target variable: `y`
- Binary classification: yes / no
- Exclude `duration` from the primary model due to data leakage.
- Preserve the original raw dataset.
- Fit preprocessing transformations only on training data.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [3]:
project_root = Path.cwd()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "bank-additional-full.csv"

df = pd.read_csv(data_path, sep=";")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.10,93.99,-36.40,4.86,5191.00,no


In [4]:
df_work = df.copy()

print("Original dataset shape:", df.shape)
print("Working dataset shape:", df_work.shape)

Original dataset shape: (41188, 21)
Working dataset shape: (41188, 21)


## 1. Duplicate Record Investigation

During EDA, 12 duplicate records were identified.
Before removing them, we will inspect the duplicate observations
to determine whether they are exact duplicates.

In [5]:
duplicate_count = df_work.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 12


In [6]:
duplicate_rows = df_work[df_work.duplicated(keep=False)]

print("Total rows involved in duplication:", len(duplicate_rows))

duplicate_rows.sort_values(
    by=list(df_work.columns)
).head(30)

Total rows involved in duplication: 24


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
28476,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.80,93.08,-47.10,1.42,5099.10,no
28477,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.80,93.08,-47.10,1.42,5099.10,no
14155,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.40,93.92,-42.70,4.96,5228.10,no
14234,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.40,93.92,-42.70,4.96,5228.10,no
18464,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no
18465,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no
32505,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.80,92.89,-46.20,1.31,5099.10,no
32516,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.80,92.89,-46.20,1.31,5099.10,no
12260,36,retired,married,unknown,no,no,no,telephone,jul,thu,88,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no
12261,36,retired,married,unknown,no,no,no,telephone,jul,thu,88,1,999,0,nonexistent,1.40,93.92,-42.70,4.97,5228.10,no


## 2. Removing Duplicate Records

The initial dataset contained 12 duplicate occurrences,
representing 12 pairs of exact duplicate records.

Since these records were identical across all available columns,
we retain the first occurrence and remove the repeated copies.

The original dataset remains unchanged.

In [7]:
# Remove exact duplicate records
df_work = df_work.drop_duplicates(keep="first").copy()

print("Shape after removing duplicates:", df_work.shape)
print("Remaining duplicate records:", df_work.duplicated().sum())

Shape after removing duplicates: (41176, 21)
Remaining duplicate records: 0


In [8]:
print("Original dataset shape:", df.shape)
print("Cleaned working dataset shape:", df_work.shape)

Original dataset shape: (41188, 21)
Cleaned working dataset shape: (41176, 21)


## 3. Investigation of Unknown Values

The dataset contains unknown categories in several categorical
features. We will examine their frequency and relationship with
the target variable before deciding how to handle them.

In [9]:
# Identify categorical columns
categorical_columns = df_work.select_dtypes(
    include=["object", "str"]
).columns

# Count unknown values in each categorical column
unknown_counts = (
    df_work[categorical_columns]
    .eq("unknown")
    .sum()
)

# Calculate percentages
unknown_percentage = (
    unknown_counts / len(df_work) * 100
)

unknown_summary = pd.DataFrame({
    "unknown_count": unknown_counts,
    "unknown_percentage": unknown_percentage
})

unknown_summary = unknown_summary[
    unknown_summary["unknown_count"] > 0
].sort_values(
    by="unknown_count",
    ascending=False
)

unknown_summary.round(2)

,unknown_count,unknown_percentage
default,8596,20.88
education,1730,4.20
housing,990,2.40
loan,990,2.40
job,330,0.80
marital,80,0.19


In [10]:
unknown_target_analysis = {}

for col in unknown_summary.index:

    analysis = df_work.groupby(
        df_work[col].eq("unknown").map({
            True: "unknown",
            False: "known"
        }),
        observed=True
    )["y"].agg(
        total_customers="count",
        subscriptions=lambda x: (x == "yes").sum()
    )

    analysis["subscription_rate"] = (
        analysis["subscriptions"] /
        analysis["total_customers"] * 100
    )

    unknown_target_analysis[col] = analysis.round(2)

for col, result in unknown_target_analysis.items():
    print(f"\nSubscription analysis: {col}")
    display(result)


Subscription analysis: default


,total_customers,subscriptions,subscription_rate
default,,,
known,32580,4196,12.88
unknown,8596,443,5.15



Subscription analysis: education


,total_customers,subscriptions,subscription_rate
education,,,
known,39446,4388,11.12
unknown,1730,251,14.51



Subscription analysis: housing


,total_customers,subscriptions,subscription_rate
housing,,,
known,40186,4532,11.28
unknown,990,107,10.81


... (2 outputs left out)


Subscription analysis: job


,total_customers,subscriptions,subscription_rate
job,,,
known,40846,4602,11.27
unknown,330,37,11.21



Subscription analysis: marital


,total_customers,subscriptions,subscription_rate
marital,,,
known,41096,4627,11.26
unknown,80,12,15.00


### 3.1 Decision: Handling Unknown Categories

Unknown values were found in six categorical features:
default, education, housing, loan, job and marital.

Their subscription rates were compared against records with
known information.

The unknown categories will be retained rather than replaced
with the most frequent category or removed.

This preserves the distinction between unavailable information
and an explicitly recorded category.

The marital unknown category has a relatively small sample size,
so its observed subscription rate should be interpreted cautiously.

## 4. Investigation of Special Values

The pdays feature uses 999 as a special indicator for customers
who were not previously contacted.

Similarly, poutcome contains the category 'nonexistent' for
customers without a previous campaign outcome.

These values represent meaningful business information and
will not be treated as ordinary missing values.

In [11]:
print("pdays value distribution:")
display(df_work["pdays"].value_counts().head(10))

print("\npoutcome value distribution:")
display(df_work["poutcome"].value_counts())

pdays value distribution:


pdays
999    39661
3        439
6        412
4        118
9         64
2         61
7         60
12        58
10        52
5         46
Name: count, dtype: int64


poutcome value distribution:


poutcome
nonexistent    35551
failure         4252
success         1373
Name: count, dtype: int64

In [12]:
missing_summary = pd.DataFrame({
    "missing_count": df_work.isnull().sum(),
    "missing_percentage": (
        df_work.isnull().mean() * 100
    )
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]

missing_summary

,missing_count,missing_percentage


### 4.1 pdays

The pdays feature represents the number of days since the
customer was last contacted during a previous campaign.

A value of 999 indicates that the customer was not previously
contacted. This is a special indicator rather than an actual
elapsed duration of 999 days.

In [13]:
print("pdays distribution:")
display(
    df_work["pdays"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nNumber of records with pdays = 999:")
print((df_work["pdays"] == 999).sum())

print("\nPercentage of records with pdays = 999:")
print(round((df_work["pdays"] == 999).mean() * 100, 2), "%")

pdays distribution:


pdays
0      15
1      26
2      61
3     439
4     118
5      46
6     412
7      60
8      18
9      64
10     52
11     28
12     58
13     36
14     20
15     24
16     11
17      8
18      7
19      3
Name: count, dtype: int64


Number of records with pdays = 999:
39661

Percentage of records with pdays = 999:
96.32 %


In [14]:
print("poutcome distribution:")

display(
    df_work["poutcome"].value_counts()
)

poutcome distribution:


poutcome
nonexistent    35551
failure         4252
success         1373
Name: count, dtype: int64

In [15]:
feature_summary = pd.DataFrame({
    "Data Type": df_work.dtypes.astype(str),
    "Unique Values": df_work.nunique(),
    "Missing Values": df_work.isnull().sum()
})

display(feature_summary)

,Data Type,Unique Values,Missing Values
age,int64,78,0
job,str,12,0
marital,str,4,0
education,str,8,0
default,str,3,0
housing,str,3,0
loan,str,3,0
contact,str,2,0
month,str,10,0
day_of_week,str,5,0


## 5. Feature Type Identification and Preparation

### 5.1 Identifying Numerical and Categorical Features

The dataset contains both numerical and categorical predictors.
Identifying these feature types will help us apply appropriate
preprocessing techniques later.

In [16]:
numerical_features = df_work.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = df_work.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:")
print(numerical_features)

print("\nCategorical columns:")
print(categorical_features)

print("\nNumber of numerical columns:", len(numerical_features))
print("Number of categorical columns:", len(categorical_features))

Numerical columns:
['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Categorical columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome', 'y']

Number of numerical columns: 10
Number of categorical columns: 11


C:\Users\Administrator\AppData\Local\Temp\ipykernel_19548\2632928189.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df_work.select_dtypes(include=["object"]).columns.tolist()


### 5.2 Separating Target and Predictor Variables

The target variable is `y`, which indicates whether a customer
subscribed to a term deposit.

All remaining columns are initially treated as potential
predictors. The target will be separated before model training
to prevent it from being included in the input features.

In [17]:
X = df_work.drop(columns=["y"])
y = df_work["y"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Predictor shape: (41176, 20)
Target shape: (41176,)

Target distribution:
y
no     36537
yes     4639
Name: count, dtype: int64


### 5.3 Excluding Duration to Prevent Data Leakage

The `duration` feature represents the duration of the last
contact with the customer, measured in seconds.

Since this information is only available after the call has
taken place, using it would introduce information leakage
when predicting subscription before contacting the customer.

Therefore, `duration` is excluded from the primary model.

In [18]:
X = X.drop(columns=["duration"])

print("Predictor shape after excluding duration:", X.shape)
print("\nRemaining features:")
print(X.columns.tolist())

Predictor shape after excluding duration: (41176, 19)

Remaining features:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


### 5.4 Feature Engineering: Previous Contact Indicator

The `pdays` column uses 999 to indicate that a customer was
not previously contacted.

A binary feature named `previously_contacted` is created to
explicitly capture whether a customer had been contacted
in a previous campaign.

The special value 999 in `pdays` will subsequently be treated
as missing, while the new indicator preserves the information
about previous contact.

In [19]:
X["previously_contacted"] = (X["pdays"] != 999).astype(int)

print("Previously contacted distribution:")
print(X["previously_contacted"].value_counts())

print("\nSubscription rate by previous contact:")
print(
    pd.DataFrame({
        "Total": X.groupby("previously_contacted").size(),
        "Subscribed": y.groupby(X["previously_contacted"]).apply(
            lambda values: (values == "yes").sum()
        ),
        "Subscription Rate (%)": y.groupby(X["previously_contacted"]).apply(
            lambda values: (values == "yes").mean() * 100
        )
    }).round(2)
)

Previously contacted distribution:
previously_contacted
0    39661
1     1515
Name: count, dtype: int64

Subscription rate by previous contact:
                      Total  Subscribed  Subscription Rate (%)
previously_contacted                                          
0                     39661        3672                   9.26
1                      1515         967                  63.83


In [20]:
X["pdays"] = X["pdays"].replace(999, np.nan)

print("Missing values in pdays after handling 999:")
print(X["pdays"].isnull().sum())

print("\nPreviously contacted feature:")
print(X["previously_contacted"].value_counts())

Missing values in pdays after handling 999:
39661

Previously contacted feature:
previously_contacted
0    39661
1     1515
Name: count, dtype: int64


### 5.5 Validating the Previous Contact Feature

The newly created `previously_contacted` feature is compared
with the existing `previous` and `poutcome` columns.

This validation helps check whether the engineered feature
is logically consistent with the original dataset.

In [21]:
print("Previously contacted vs previous contacts:")
display(pd.crosstab(
    X["previously_contacted"],
    X["previous"]
))

print("\nPreviously contacted vs previous campaign outcome:")
display(pd.crosstab(
    X["previously_contacted"],
    X["poutcome"]
))

Previously contacted vs previous contacts:


previous,0,1,2,3,4,5,6,7
previously_contacted,,,,,,,,
0,35551,3696,349,50,12,2,1,0
1,0,865,405,166,58,16,4,1



Previously contacted vs previous campaign outcome:


poutcome,failure,nonexistent,success
previously_contacted,,,
0,4110,35551,0
1,142,0,1373


In [22]:
mismatch_records = df_work[
    (df_work["pdays"] == 999) &
    (
        (df_work["previous"] > 0) |
        (df_work["poutcome"] != "nonexistent")
    )
]

print("Number of inconsistent records:", len(mismatch_records))

display(
    mismatch_records[
        ["pdays", "previous", "poutcome", "y"]
    ].head(10)
)

Number of inconsistent records: 4110


,pdays,previous,poutcome,y
24013,999,1,failure,no
24019,999,1,failure,yes
24076,999,1,failure,no
24102,999,1,failure,yes
24113,999,1,failure,no
24124,999,1,failure,no
24135,999,1,failure,no
24139,999,1,failure,no
24140,999,1,failure,no
24143,999,1,failure,no


In [23]:
print("Mismatch combinations:")

display(
    pd.crosstab(
        mismatch_records["previous"],
        mismatch_records["poutcome"]
    )
)

Mismatch combinations:


poutcome,failure
previous,
1,3696
2,349
3,50
4,12
5,2
6,1


In [24]:
X = X.rename(
    columns={"previously_contacted": "pdays_contact_recorded"}
)

print(X.columns.tolist())

['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'pdays_contact_recorded']


### Data Quality Observation: pdays, previous and poutcome

A cross-tabulation revealed 4,110 records where:

- pdays = 999
- previous > 0
- poutcome = failure

Although the dataset documentation defines 999 as an indicator
that the customer was not previously contacted, these records
also contain previous contact counts and a failed campaign
outcome.

The exact reason for this discrepancy cannot be established
from the available dataset documentation.

These records have been retained without modification.
The engineered feature is therefore named
`pdays_contact_recorded` to describe exactly what it captures.

### 5.6 Final Feature Type Identification

After feature engineering and exclusion of `duration`,
the numerical and categorical predictor columns are
identified again.

These feature lists will be used to construct separate
preprocessing pipelines.

In [25]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal numerical features:", len(numerical_features))
print("Total categorical features:", len(categorical_features))
print("Total predictors:", X.shape[1])

Numerical features:
['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'pdays_contact_recorded']

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

Total numerical features: 10
Total categorical features: 10
Total predictors: 20


C:\Users\Administrator\AppData\Local\Temp\ipykernel_19548\342599137.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


### 5.7 Train-Test Split

The dataset is divided into training and testing subsets
using an 80:20 split.

Stratified sampling is applied to preserve the original
class distribution of the target variable in both subsets.

A fixed random state ensures reproducibility.

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training predictor shape:", X_train.shape)
print("Testing predictor shape:", X_test.shape)

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nTraining target distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training predictor shape: (32940, 20)
Testing predictor shape: (8236, 20)

Training target shape: (32940,)
Testing target shape: (8236,)

Training target distribution (%):
y
no    88.73
yes   11.27
Name: proportion, dtype: float64

Testing target distribution (%):
y
no    88.73
yes   11.27
Name: proportion, dtype: float64


## 6. Preprocessing Pipeline 

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

### 6.1 Numerical Pipeline 

In [38]:
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### 6.2 Categorical Pipeline 

In [39]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

### 6.3 Combining Both Pipelines 

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [41]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (32940, 63)
Processed testing shape: (8236, 63)


### 6.4 Validating Processed Datasets

The transformed training and testing datasets are checked
for their dimensions, missing values and numerical validity.

This ensures that the preprocessing pipeline has successfully
prepared the data for machine learning.

In [42]:
from scipy.sparse import issparse

print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)

print("\nTraining data contains missing values:",
      np.isnan(X_train_processed.data if issparse(X_train_processed)
               else X_train_processed).any())

print("Testing data contains missing values:",
      np.isnan(X_test_processed.data if issparse(X_test_processed)
               else X_test_processed).any())

print("\nTraining data type:", type(X_train_processed))
print("Testing data type:", type(X_test_processed))

Training data shape: (32940, 63)
Testing data shape: (8236, 63)

Training data contains missing values: False
Testing data contains missing values: False

Training data type: <class 'numpy.ndarray'>
Testing data type: <class 'numpy.ndarray'>


In [43]:
processed_feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(processed_feature_names))

print("\nFirst 30 processed feature names:")
print(processed_feature_names[:30])

Total processed features: 63

First 30 processed feature names:
['num__age' 'num__campaign' 'num__pdays' 'num__previous'
 'num__emp.var.rate' 'num__cons.price.idx' 'num__cons.conf.idx'
 'num__euribor3m' 'num__nr.employed' 'num__pdays_contact_recorded'
 'cat__job_admin.' 'cat__job_blue-collar' 'cat__job_entrepreneur'
 'cat__job_housemaid' 'cat__job_management' 'cat__job_retired'
 'cat__job_self-employed' 'cat__job_services' 'cat__job_student'
 'cat__job_technician' 'cat__job_unemployed' 'cat__job_unknown'
 'cat__marital_divorced' 'cat__marital_married' 'cat__marital_single'
 'cat__marital_unknown' 'cat__education_basic.4y'
 'cat__education_basic.6y' 'cat__education_basic.9y'
 'cat__education_high.school']


In [44]:
print("Numerical features before transformation:",
      len(numerical_features))

print("Categorical features before transformation:",
      len(categorical_features))

print("Total features after transformation:",
      len(processed_feature_names))

Numerical features before transformation: 10
Categorical features before transformation: 10
Total features after transformation: 63


## 7. Preprocessing Summary

The following preprocessing operations were completed:

1. Removed 12 exact duplicate records.
2. Retained unknown categorical values as separate categories.
3. Preserved the special meaning of the pdays variable.
4. Created the pdays_contact_recorded engineered feature.
5. Excluded duration to prevent information leakage.
6. Separated the target variable from the predictors.
7. Performed an 80:20 stratified train-test split.
8. Applied median imputation and standardization to numerical features.
9. Applied most-frequent imputation and one-hot encoding to categorical features.
10. Fitted preprocessing transformations exclusively on the training data.

The resulting training and testing datasets are ready
for machine learning model development.